# EndoWherAI — Poster Diagram Generator
Generates all visual assets needed for the research poster.
Run cells in order. All outputs saved to `../outputs/`.

In [21]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay
)
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = os.path.abspath('')
BASE_DIR     = os.path.dirname(NOTEBOOK_DIR)
DATA_DIR     = os.path.join(BASE_DIR, 'data')
OUT_DIR      = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# Brand colours
PRIMARY  = '#854a58'
ACCENT   = '#bf969d'
ACCENT2  = '#a46e76'
SUCCESS  = '#90aa8d'
LIGHT    = '#ffeef2'
DARK     = '#2d1a1f'

print('Output dir:', OUT_DIR)

Output dir: /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs


## 1 — Load data & model

In [22]:
# ── Load cleaned dataset & reproduce the same 80/20 split ──────────────────
df = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_endowher_research.csv'))
df_smote = pd.read_csv(os.path.join(DATA_DIR, 'expanded_endowher_smoteenn.csv'))

feature_cols = [c for c in df.columns if c != 'has_condition']
X_all = df[feature_cols].values
y_all = df['has_condition'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=42
)

# ── Load saved model ────────────────────────────────────────────────────────
bundle = joblib.load(os.path.join(OUT_DIR, 'endowher_stacking_model.joblib'))
model  = bundle['model']

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

ACC = accuracy_score(y_test, y_pred)
F1  = f1_score(y_test, y_pred, average='weighted')
AUC = roc_auc_score(y_test, y_prob)

print(f'Holdout  Accuracy : {ACC:.4f}')
print(f'Holdout  F1 Score : {F1:.4f}')
print(f'Holdout  ROC-AUC  : {AUC:.4f}')
print(f'\nRaw N={len(df)}  |  SMOTEENN N={len(df_smote)}  |  Test N={len(X_test)}')

Holdout  Accuracy : 0.8571
Holdout  F1 Score : 0.8569
Holdout  ROC-AUC  : 0.9248

Raw N=175  |  SMOTEENN N=542  |  Test N=35


## 2 — ROC Curve

In [23]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.set_facecolor('none')
fig.patch.set_facecolor('none')

ax.plot(fpr, tpr, color=PRIMARY, lw=2.5, label=f'Stacking Ensemble  (AUC = {AUC:.3f})')
ax.plot([0,1],[0,1], '--', color='#cccccc', lw=1.2, label='Random classifier (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.12, color=PRIMARY)

ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.05)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — EndoWherAI Stacking Ensemble', fontsize=13, fontweight='bold', color=DARK)
ax.legend(loc='lower right', fontsize=10, framealpha=0.9)
ax.grid(alpha=0.25)
ax.spines[['top','right']].set_visible(False)

# annotate AUC
ax.annotate(f'AUC = {AUC:.3f}', xy=(0.6, 0.45),
            fontsize=14, fontweight='bold', color=PRIMARY,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT, edgecolor=ACCENT, alpha=0.9))

plt.tight_layout()
path = os.path.join(OUT_DIR, 'roc_curve.png')
plt.savefig(path, dpi=180, bbox_inches='tight', transparent=True)
plt.close()
print('Saved →', path)

Saved → /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs/roc_curve.png


## 3 — Confusion Matrix

In [24]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5.5, 5))
fig.patch.set_facecolor('none')
ax.set_facecolor('none')

im = ax.imshow(cm, cmap='RdPu', aspect='auto')

labels = ['No Condition', 'Has Condition']
ax.set_xticks([0, 1]); ax.set_xticklabels(labels, fontsize=11)
ax.set_yticks([0, 1]); ax.set_yticklabels(labels, fontsize=11, rotation=90, va='center')
ax.set_xlabel('Predicted label', fontsize=12, labelpad=8)
ax.set_ylabel('True label', fontsize=12, labelpad=8)
ax.set_title('Confusion Matrix — Holdout Test Set', fontsize=13, fontweight='bold', color=DARK)

thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=24, fontweight='bold',
                color='white' if cm[i, j] > thresh else DARK)

# labels for quadrants
quad = {(0,0):'TN', (0,1):'FP', (1,0):'FN', (1,1):'TP'}
for (i,j), lbl in quad.items():
    ax.text(j+0.4, i-0.38, lbl, ha='right', va='top',
            fontsize=9, color='white' if cm[i,j] > thresh else DARK, alpha=0.7)

plt.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
plt.tight_layout()
path = os.path.join(OUT_DIR, 'confusion_matrix.png')
plt.savefig(path, dpi=180, bbox_inches='tight', transparent=True)
plt.close()
print('Saved →', path)

Saved → /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs/confusion_matrix.png


## 4 — Class Distribution Before vs After SMOTEENN

In [25]:
raw_counts   = pd.Series(y_all).value_counts().sort_index()       # 175 total
smote_counts = df_smote['has_condition'].value_counts().sort_index()  # 542 total

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), sharey=False)
fig.patch.set_facecolor('none')

for ax, counts, title, total in [
    (axes[0], raw_counts,   f'Before SMOTEENN\n(N = {int(raw_counts.sum())})',   int(raw_counts.sum())),
    (axes[1], smote_counts, f'After SMOTEENN\n(N = {int(smote_counts.sum())})', int(smote_counts.sum())),
]:
    bars = ax.bar(
        ['No Condition\n(Class 0)', 'Has Condition\n(Class 1)'],
        counts.values,
        color=[SUCCESS, PRIMARY], width=0.55, edgecolor='white', linewidth=1.5
    )
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
                f'{v}\n({v/total:.0%})',
                ha='center', va='bottom', fontsize=11, fontweight='bold', color=DARK)
    ax.set_title(title, fontsize=12, fontweight='bold', color=DARK)
    ax.set_ylabel('Sample count', fontsize=10)
    ax.set_ylim(0, max(counts.values) * 1.22)
    ax.spines[['top','right']].set_visible(False)
    ax.set_facecolor('none')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Class Distribution — Original Dataset vs SMOTEENN Resampled', 
             fontsize=13, fontweight='bold', color=DARK, y=1.02)
plt.tight_layout()
path = os.path.join(OUT_DIR, 'class_distribution.png')
plt.savefig(path, dpi=180, bbox_inches='tight', transparent=True)
plt.close()
print('Saved →', path)

Saved → /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs/class_distribution.png


## 5 — Performance Metrics Summary Bar

In [26]:
# Known CV values from the pipeline run (update these if you re-run the pipeline)
# These come from the printed output of endowher_pipeline.py
CV_ACC  = 0.8821   # update from your pipeline output
CV_F1   = 0.8798   # update from your pipeline output

metrics = {
    'CV Accuracy\n(10-fold × 100 rep)': CV_ACC,
    'CV F1 Score\n(10-fold × 100 rep)': CV_F1,
    'Holdout\nAccuracy':               ACC,
    'Holdout\nF1 Score':               F1,
    'ROC-AUC\n(holdout)':              AUC,
}

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('none')
ax.set_facecolor('none')

colors = [ACCENT, ACCENT, PRIMARY, PRIMARY, SUCCESS]
bars = ax.bar(list(metrics.keys()), list(metrics.values()),
              color=colors, width=0.55, edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{val:.3f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold', color=DARK)

# 90% target line
ax.axhline(0.90, color='#e74c3c', lw=1.5, ls='--', label='90% target')
ax.text(4.55, 0.901, '90% target', color='#e74c3c', fontsize=9, va='bottom')

ax.set_ylim(0.70, 1.04)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('EndoWherAI — Model Performance Summary', fontsize=13, fontweight='bold', color=DARK)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', labelsize=10)

legend_patches = [
    mpatches.Patch(color=ACCENT, label='Cross-validation'),
    mpatches.Patch(color=PRIMARY, label='Holdout test set'),
    mpatches.Patch(color=SUCCESS, label='ROC-AUC'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=10, framealpha=0.85)

plt.tight_layout()
path = os.path.join(OUT_DIR, 'performance_metrics.png')
plt.savefig(path, dpi=180, bbox_inches='tight', transparent=True)
plt.close()
print('Saved →', path)

Saved → /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs/performance_metrics.png


## 6 — Pipeline Architecture Flowchart

In [27]:
fig, ax = plt.subplots(figsize=(14, 9))
fig.patch.set_facecolor('none')
ax.set_facecolor('none')
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')

def box(ax, x, y, w, h, label, sublabel='', color=PRIMARY, text_color='white',
        fontsize=10, radius=0.25):
    rect = FancyBboxPatch((x - w/2, y - h/2), w, h,
                           boxstyle=f'round,pad={radius}',
                           facecolor=color, edgecolor='white',
                           linewidth=1.8, zorder=3)
    ax.add_patch(rect)
    if sublabel:
        ax.text(x, y + 0.15, label, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color=text_color, zorder=4)
        ax.text(x, y - 0.22, sublabel, ha='center', va='center',
                fontsize=7.5, color=text_color, alpha=0.85, zorder=4)
    else:
        ax.text(x, y, label, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color=text_color, zorder=4)

def arrow(ax, x1, y1, x2, y2, color='#aaaaaa'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color,
                                lw=1.8, connectionstyle='arc3,rad=0.0'),
                zorder=2)

def label_arrow(ax, x, y, text, color='#777777'):
    ax.text(x, y, text, ha='center', va='center', fontsize=8,
            color=color, style='italic', zorder=5)

# ─── SECTION BACKGROUNDS ─────────────────────────────────────────────────────
sections = [
    (0.1, 5.2, 3.7, 3.55, '#fff0f3', 'DATA LAYER', PRIMARY),
    (3.95, 5.2, 6.1, 3.55, '#f5f0ff', 'ML PIPELINE', '#5a3480'),
    (10.2, 5.2, 3.65, 3.55, '#f0f8f0', 'EXPLAINABILITY', SUCCESS),
    (0.1, 1.4, 5.0, 3.3, '#fff8e0', 'BACKEND — FastAPI', '#856010'),
    (5.3, 1.4, 8.65, 3.3, '#e8f0ff', 'FRONTEND — Next.js 14', '#1a3a80'),
]
for (x, y, w, h, fc, lbl, lc) in sections:
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                           facecolor=fc, edgecolor=lc, linewidth=1.2,
                           alpha=0.55, zorder=1)
    ax.add_patch(rect)
    ax.text(x + 0.12, y + h - 0.22, lbl, fontsize=8.5, fontweight='bold',
            color=lc, alpha=0.9, zorder=2)

# ─── DATA LAYER ──────────────────────────────────────────────────────────────
box(ax, 1.05, 8.0, 1.7, 0.6, 'Survey Data', 'EN + BS + TR', color='#c0757e')
arrow(ax, 1.75, 8.0, 2.3, 8.0)
box(ax, 2.85, 8.0, 1.1, 0.6, 'Clean &\nEncode', color='#a4606a', fontsize=9)
arrow(ax, 3.4, 8.0, 3.85, 8.0)
box(ax, 1.05, 6.5, 1.7, 0.6, 'N = 175 rows', '93 features', color='#854a58')
arrow(ax, 1.05, 6.2, 1.05, 5.75)
box(ax, 1.05, 5.45, 1.7, 0.55, 'SMOTEENN', '→ N = 542', color=ACCENT2)
# connect clean to N=175
arrow(ax, 2.85, 7.7, 1.05, 6.8)

# ─── ML PIPELINE ─────────────────────────────────────────────────────────────
# 80/20 split
box(ax, 5.0, 8.0, 1.5, 0.58, '80/20 Split', 'stratified', color='#6a3d7a', fontsize=9)
arrow(ax, 3.85, 8.0, 4.25, 8.0)

# RF
box(ax, 6.6, 8.3, 1.55, 0.5, 'Random\nForest', 'n=300 trees', color='#7b4fa3', fontsize=9)
# XGB
box(ax, 6.6, 7.65, 1.55, 0.5, 'XGBoost', 'n=300, lr=0.05', color='#6a3d7a', fontsize=9)
arrow(ax, 5.75, 8.1, 5.83, 8.23)
arrow(ax, 5.75, 7.9, 5.83, 7.73)
label_arrow(ax, 5.78, 8.35, 'L0', color='#6a3d7a')

# Meta LR
box(ax, 8.5, 8.0, 1.5, 0.58, 'Logistic\nRegression', 'L1 meta-learner', color='#4d2b6e', fontsize=9)
arrow(ax, 7.38, 8.3, 7.75, 8.1)
arrow(ax, 7.38, 7.65, 7.75, 7.9)
label_arrow(ax, 7.7, 8.35, 'L1', color='#4d2b6e')

# CV box
box(ax, 6.6, 6.5, 3.0, 0.62, 'RepeatedStratifiedKFold', '10 folds × 100 repeats = 1,000 fits', color='#3d1f5a', fontsize=8.5)
arrow(ax, 8.5, 7.71, 8.5, 6.81)

# ─── EXPLAINABILITY ──────────────────────────────────────────────────────────
box(ax, 11.0, 8.15, 1.55, 0.55, 'SHAP', 'KernelExplainer', color='#3a7a3a', fontsize=9)
box(ax, 11.0, 7.45, 1.55, 0.55, 'LIME', 'LimeTabular', color='#2d6b3a', fontsize=9)
arrow(ax, 9.25, 8.1, 10.22, 8.15)
arrow(ax, 9.25, 7.9, 10.22, 7.45)
label_arrow(ax, 9.75, 8.35, 'predict_proba', color='#555')

box(ax, 12.7, 7.8, 1.3, 0.95, '📊 Global\nfeature\nranking', color=SUCCESS, fontsize=8.5)
box(ax, 12.7, 6.6, 1.3, 0.85, '📋 Local\npatient\nexplanation', color='#5a9a5a', fontsize=8.5)
arrow(ax, 11.78, 8.15, 12.05, 7.95)
arrow(ax, 11.78, 7.45, 12.05, 7.0)

# ─── BACKEND ─────────────────────────────────────────────────────────────────
box(ax, 1.35, 3.5, 1.85, 0.6, 'FastAPI', '/predict /chat /community', color='#9a7020', fontsize=9)
box(ax, 3.6, 3.5, 1.7, 0.6, 'Supabase', 'PostgreSQL + RLS', color='#7a5510', fontsize=9)
arrow(ax, 2.28, 3.5, 2.75, 3.5)
arrow(ax, 3.5, 3.9, 3.5, 6.5, color='#ccaa44')  # model → api
label_arrow(ax, 3.2, 5.2, 'saved model', color='#9a7020')

# ─── FRONTEND ────────────────────────────────────────────────────────────────
frontend_items = [
    (6.0, 3.8, 'Diary\nPage', '#3050a0'),
    (7.4, 3.8, 'Cycle\nTracker', '#3050a0'),
    (8.8, 3.8, 'Weekly\nCheck-in', '#3050a0'),
    (10.2, 3.8, 'AI Risk\nInsights', '#1a3a80'),
    (11.6, 3.8, 'Remedies\n& Chat', '#1a3a80'),
    (13.0, 3.8, 'Education', '#1a3a80'),
]
for (x, y, lbl, col) in frontend_items:
    box(ax, x, y, 1.15, 0.65, lbl, color=col, fontsize=8)

# next.js bar
box(ax, 9.5, 2.2, 7.8, 0.55, 'Next.js 14  ·  TypeScript  ·  Tailwind CSS v4  ·  Supabase Auth',
    color='#1a3a80', fontsize=9.5)

# API ↔ Frontend arrows
arrow(ax, 4.45, 3.5, 5.45, 3.5)
label_arrow(ax, 4.95, 3.7, 'REST JSON', color='#555')

# ─── TITLE ───────────────────────────────────────────────────────────────────
ax.text(7.0, 0.85, 'EndoWherAI — System Architecture',
        ha='center', va='center', fontsize=15, fontweight='bold', color=DARK)
ax.text(7.0, 0.42, 'PCOS & Endometriosis Privacy-First Symptom Tracking Platform with Stacking Ensemble ML + SHAP/LIME Explainability',
        ha='center', va='center', fontsize=9.5, color='#555', style='italic')

plt.tight_layout()
path = os.path.join(OUT_DIR, 'architecture_diagram.png')
plt.savefig(path, dpi=200, bbox_inches='tight', transparent=True)
plt.close()
print('Saved →', path)

Saved → /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs/architecture_diagram.png


## 7 — Tech Stack Visual

In [28]:
fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor('none')
ax.set_facecolor('none')
ax.set_xlim(0, 13)
ax.set_ylim(0, 7)
ax.axis('off')

categories = [
    {
        'title': 'Frontend',
        'color': '#1a3a80',
        'bg': '#e8f0ff',
        'x': 0.2, 'y': 3.6, 'w': 3.5, 'h': 3.1,
        'items': [
            ('Next.js 14', 'App Router + SSR'),
            ('TypeScript', 'Type-safe components'),
            ('Tailwind CSS v4', 'Responsive design'),
            ('Supabase JS', 'Auth + realtime'),
            ('Recharts', 'Data visualisation'),
        ]
    },
    {
        'title': 'Backend / API',
        'color': '#8a5500',
        'bg': '#fff8e0',
        'x': 3.9, 'y': 3.6, 'w': 3.0, 'h': 3.1,
        'items': [
            ('FastAPI', 'REST endpoints'),
            ('Python 3.11', 'Runtime'),
            ('Supabase', 'PostgreSQL + RLS'),
            ('Pydantic v2', 'Schema validation'),
            ('JWT / Bearer', 'Auth middleware'),
        ]
    },
    {
        'title': 'Machine Learning',
        'color': '#4d1f6e',
        'bg': '#f3eeff',
        'x': 7.1, 'y': 3.6, 'w': 3.2, 'h': 3.1,
        'items': [
            ('scikit-learn', 'RF + Stacking + CV'),
            ('XGBoost', 'Gradient boosting L0'),
            ('imbalanced-learn', 'SMOTEENN resampling'),
            ('SHAP', 'Global explainability'),
            ('LIME', 'Local explainability'),
        ]
    },
    {
        'title': 'Privacy & Compliance',
        'color': '#1a6640',
        'bg': '#e8fff0',
        'x': 10.5, 'y': 3.6, 'w': 2.3, 'h': 3.1,
        'items': [
            ('EDPB 01/2025', 'Pseudonymisation'),
            ('GDPR Art.25', 'Privacy by design'),
            ('Row-Level Security', 'Supabase RLS'),
            ('No identifiers', 'Survey anonymity'),
        ]
    },
]

for cat in categories:
    rect = FancyBboxPatch((cat['x'], cat['y']), cat['w'], cat['h'],
                           boxstyle='round,pad=0.1',
                           facecolor=cat['bg'], edgecolor=cat['color'],
                           linewidth=2.0, zorder=1)
    ax.add_patch(rect)
    ax.text(cat['x'] + cat['w']/2, cat['y'] + cat['h'] - 0.28,
            cat['title'], ha='center', fontsize=11.5, fontweight='bold',
            color=cat['color'], zorder=3)
    ax.axhline(cat['y'] + cat['h'] - 0.5, xmin=(cat['x'])/13,
               xmax=(cat['x']+cat['w'])/13,
               color=cat['color'], lw=1.0, alpha=0.4, zorder=2)

    for i, (name, desc) in enumerate(cat['items']):
        ypos = cat['y'] + cat['h'] - 0.85 - i * 0.47
        ax.text(cat['x'] + 0.2, ypos + 0.07, name,
                fontsize=9.5, fontweight='bold', color=cat['color'], zorder=3)
        ax.text(cat['x'] + 0.2, ypos - 0.17, desc,
                fontsize=8, color='#555555', zorder=3)

# Dataset info row
data_items = [
    ('Survey Dataset', 'N=175  |  93 features  |  EN+BS+TR'),
    ('Stacking Ensemble', 'RF(300) + XGB(300) → LR(C=1)'),
    ('Validation', '10-fold × 100 repeats  =  1,000 fits'),
    ('ROC-AUC', f'{AUC:.3f}  (holdout test set, N=35)'),
]
x_pos = [0.2, 3.45, 6.7, 9.95]
colors_d = [ACCENT2, '#5a3480', '#1a3a80', SUCCESS]
for (title, val), x, col in zip(data_items, x_pos, colors_d):
    rect = FancyBboxPatch((x, 0.3), 3.05, 2.9,
                           boxstyle='round,pad=0.1',
                           facecolor='white', edgecolor=col,
                           linewidth=1.6, zorder=1)
    ax.add_patch(rect)
    ax.text(x + 1.525, 1.9, title, ha='center', fontsize=10.5,
            fontweight='bold', color=col, zorder=3)
    ax.text(x + 1.525, 1.25, val, ha='center', fontsize=9,
            color='#444', zorder=3, wrap=True)

ax.text(6.5, 6.92, 'EndoWherAI — Full Technology Stack',
        ha='center', fontsize=15, fontweight='bold', color=DARK)

plt.tight_layout()
path = os.path.join(OUT_DIR, 'tech_stack.png')
plt.savefig(path, dpi=200, bbox_inches='tight', transparent=True)
plt.close()
print('Saved →', path)

Saved → /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs/tech_stack.png


In [32]:
# ─────────────────────────────────────────────────────────────────────────────
# 7B — Research Summary Infographic (poster-ready)
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 9))
fig.patch.set_facecolor('none')
ax.set_facecolor('none')
ax.set_xlim(0, 15)
ax.set_ylim(0, 9)
ax.axis('off')

title_y = 8.6
ax.text(7.5, title_y, 'EndoWherAI — Data, ML, Explainability, and Deployment Overview',
        ha='center', va='center', fontsize=16, fontweight='bold', color=DARK)
ax.text(7.5, 8.28,
        'Google Forms survey data → stacking ensemble prediction → SHAP/LIME explanations → web deployment',
        ha='center', va='center', fontsize=9.5, color='#555555', style='italic')

def panel(x, y, w, h, title, body_lines, edge, fill, title_color=None):
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.12',
                           facecolor=fill, edgecolor=edge, linewidth=1.8, alpha=0.95, zorder=1)
    ax.add_patch(rect)
    ax.text(x + 0.22, y + h - 0.32, title, ha='left', va='top', fontsize=12,
            fontweight='bold', color=title_color or edge, zorder=3)
    for i, line in enumerate(body_lines):
        ax.text(x + 0.22, y + h - 0.72 - i * 0.34, line, ha='left', va='top',
                fontsize=9.2, color='#333333', zorder=3)

def arrow(x1, y1, x2, y2, color='#8a8a8a'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle='-|>', lw=2.0, color=color, shrinkA=0, shrinkB=0))

# Top row: data and preprocessing
panel(0.35, 5.45, 4.25, 2.2, '1. Data Collection', [
    'Manual Google Forms collection',
    '175 survey responses',
    '93 features after consolidation',
    'Three languages: EN, BS, TR',
    'Binary target: suspected / diagnosed PCOS-endo-like status',
    'Medical review confirmed by Dr. Meryem Ceyhan',
], PRIMARY, '#fff1f3')

panel(5.15, 5.45, 4.25, 2.2, '2. Preprocessing', [
    'Cleaning and column normalization',
    'Encoding multilingual responses',
    'Class rebalancing with SMOTEENN',
    'SMOTE interpolation + Edited Nearest Neighbour',
    'Preserves minority class structure while cleaning boundaries',
], ACCENT, '#f6f0ff')

panel(9.95, 5.45, 4.65, 2.2, '3. Predictive Model', [
    'Stacking ensemble classifier',
    'Random Forest (n=300) as base learner',
    'XGBoost (n=300) as base learner',
    'Logistic Regression meta-learner',
    'Outputs risk probability and categorical risk band',
], '#5a3480', '#f5efff', title_color='#4d1f6e')

panel(0.35, 2.15, 4.25, 2.35, '4. Robustness & Validation', [
    'RepeatedStratifiedKFold',
    '10 folds × 100 repeats = 1,000 fits',
    'Evaluates stability under repeated resampling',
    'Holdout metrics: Accuracy, F1, ROC-AUC',
], SUCCESS, '#eefbf1')

panel(5.15, 2.15, 4.25, 2.35, '5. Explainability', [
    'SHAP KernelExplainer for global importance',
    'LIME LimeTabular for local explanations',
    'Feature-level transparency for each result',
    'Supported by offline SHAP summaries and local case visuals',
], '#2f7a57', '#effbf6')

panel(9.95, 2.15, 4.65, 2.35, '6. Deployment Stack', [
    'Frontend: Next.js 14, TypeScript, Tailwind CSS v4',
    'Backend: FastAPI',
    'Data layer: Supabase / PostgreSQL',
    'JWT authentication + row-level security',
    'Pseudonymisation aligned with EDPB 01/2025',
], '#1a3a80', '#eef4ff')

# Center connectors and labels
arrow(4.6, 6.5, 5.15, 6.5, color=PRIMARY)
arrow(9.4, 6.5, 9.95, 6.5, color=ACCENT)
arrow(4.6, 3.2, 5.15, 3.2, color=SUCCESS)
arrow(9.4, 3.2, 9.95, 3.2, color='#1a3a80')

# Bottom strip: privacy note and poster-ready summary figures
bottom = FancyBboxPatch((0.35, 0.35), 14.25, 1.2, boxstyle='round,pad=0.12',
                         facecolor='#faf7f8', edgecolor='#c9b7bf', linewidth=1.4, alpha=0.9, zorder=1)
ax.add_patch(bottom)
ax.text(0.62, 1.23, 'Privacy and governance', fontsize=11.5, fontweight='bold', color='#7a4560', zorder=3)
ax.text(0.62, 0.9, 'No real names, birth years, or geographic identifiers stored. Pseudonymised IDs only. Supabase RLS protects user rows.',
        fontsize=9.1, color='#444444', zorder=3)

plt.tight_layout()
path = os.path.join(OUT_DIR, 'poster_summary_infographic.png')
plt.savefig(path, dpi=200, bbox_inches='tight', transparent=True)
plt.close()
print('Saved →', path)

Saved → /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs/poster_summary_infographic.png


In [34]:
# ─────────────────────────────────────────────────────────────────────────────
# 7C — Separate poster graphs for each theme
# ─────────────────────────────────────────────────────────────────────────────
def save_box_diagram(path_name, title, subtitle, panels, arrows=None, figsize=(14, 8), xlim=(0, 14), ylim=(0, 8)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor('none')
    ax.set_facecolor('none')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axis('off')
    ax.text((xlim[0] + xlim[1]) / 2, ylim[1] - 0.2, title, ha='center', va='top', fontsize=16, fontweight='bold', color=DARK)
    ax.text((xlim[0] + xlim[1]) / 2, ylim[1] - 0.55, subtitle, ha='center', va='top', fontsize=9.5, color='#555555', style='italic')

    def draw_panel(item):
        rect = FancyBboxPatch((item['x'], item['y']), item['w'], item['h'], boxstyle='round,pad=0.12',
                              facecolor=item.get('fill', '#ffffff'), edgecolor=item['edge'], linewidth=1.8, alpha=0.96)
        ax.add_patch(rect)
        ax.text(item['x'] + 0.2, item['y'] + item['h'] - 0.3, item['title'], ha='left', va='top', fontsize=11.5, fontweight='bold', color=item['edge'])
        for idx, line in enumerate(item.get('lines', [])):
            ax.text(item['x'] + 0.2, item['y'] + item['h'] - 0.7 - idx * 0.34, line, ha='left', va='top', fontsize=9.1, color='#333333')

    for panel in panels:
        draw_panel(panel)

    if arrows:
        for ar in arrows:
            ax.annotate('', xy=ar['to'], xytext=ar['from'], arrowprops=dict(arrowstyle='-|>', lw=2.0, color=ar.get('color', '#888888')))

    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, path_name)
    plt.savefig(out_path, dpi=220, bbox_inches='tight', transparent=True)
    plt.close()
    print('Saved →', out_path)

# 7C-1 Data and preprocessing overview
save_box_diagram(
    'data_preprocessing_overview.png',
    'EndoWherAI — Data Collection and Preprocessing',
    'Survey acquisition, medical review, cleaning, encoding, and SMOTEENN rebalancing',
    [
        {'x': 0.45, 'y': 4.55, 'w': 3.65, 'h': 2.2, 'edge': PRIMARY, 'fill': '#fff1f3', 'title': 'Collected data', 'lines': [
            '175 Google Forms responses',
            '93 consolidated features',
            '3 languages: EN, BS, TR',
            'Binary diagnosis target',
            'Reviewed by Dr. Meryem Ceyhan',
        ]},
        {'x': 4.55, 'y': 4.55, 'w': 3.65, 'h': 2.2, 'edge': ACCENT2, 'fill': '#f6f0ff', 'title': 'Cleaning + encoding', 'lines': [
            'Manual cleaning and harmonisation',
            'Text standardisation and label mapping',
            'Ordinal encoding for survey responses',
            'Dataset prepared for ML input',
        ]},
        {'x': 8.65, 'y': 4.55, 'w': 4.9, 'h': 2.2, 'edge': SUCCESS, 'fill': '#eefbf1', 'title': 'SMOTEENN rebalance', 'lines': [
            'SMOTE interpolation generates minority samples',
            'Edited Nearest Neighbour removes boundary noise',
            'Helps reduce class imbalance',
            'Improves training stability',
        ]},
        {'x': 0.9, 'y': 1.25, 'w': 12.55, 'h': 1.45, 'edge': '#6c5f66', 'fill': '#faf7f8', 'title': 'Privacy controls', 'lines': [
            'No real names, birth years, or geographic identifiers stored',
            'Pseudonymised data only, aligned with EDPB 01/2025 guidance',
            'Supabase row-level security protects each user’s records',
        ]},
    ],
    arrows=[
        {'from': (4.1, 5.65), 'to': (4.55, 5.65), 'color': PRIMARY},
        {'from': (8.2, 5.65), 'to': (8.65, 5.65), 'color': ACCENT2},
    ],
    figsize=(15, 8),
    xlim=(0, 15),
    ylim=(0, 8),
)

# 7C-2 Model architecture overview
save_box_diagram(
    'model_architecture_overview.png',
    'EndoWherAI — Stacking Ensemble Model',
    'Base learners, meta-learner, and validation design',
    [
        {'x': 0.45, 'y': 4.65, 'w': 3.15, 'h': 1.95, 'edge': '#7b4fa3', 'fill': '#f5efff', 'title': 'Base learner 1', 'lines': [
            'Random Forest',
            'n = 300 trees',
            'Captures nonlinear interactions',
        ]},
        {'x': 0.45, 'y': 1.95, 'w': 3.15, 'h': 1.95, 'edge': '#6a3d7a', 'fill': '#f3eeff', 'title': 'Base learner 2', 'lines': [
            'XGBoost',
            'n = 300 boosting rounds',
            'Learns strong tabular patterns',
        ]},
        {'x': 4.15, 'y': 3.3, 'w': 3.25, 'h': 2.0, 'edge': '#4d2b6e', 'fill': '#efe8ff', 'title': 'Meta-learner', 'lines': [
            'Logistic Regression',
            'Combines base predictions',
            'Produces final probability',
        ]},
        {'x': 8.05, 'y': 4.6, 'w': 5.05, 'h': 2.15, 'edge': DARK, 'fill': '#faf7f8', 'title': 'Training strategy', 'lines': [
            'RepeatedStratifiedKFold',
            '10 folds × 100 repeats',
            '1,000 model fits for robustness',
            'Holdout test set kept separate',
        ]},
    ],
    arrows=[
        {'from': (3.6, 5.6), 'to': (4.15, 4.2), 'color': '#7b4fa3'},
        {'from': (3.6, 2.95), 'to': (4.15, 3.15), 'color': '#6a3d7a'},
        {'from': (7.4, 4.25), 'to': (8.05, 4.25), 'color': '#4d2b6e'},
    ],
    figsize=(14, 8),
    xlim=(0, 14),
    ylim=(0, 8),
)

# 7C-3 Validation and explainability overview
save_box_diagram(
    'validation_explainability_overview.png',
    'EndoWherAI — Validation and Explainability',
    'How the model was evaluated and interpreted',
    [
        {'x': 0.5, 'y': 4.65, 'w': 3.8, 'h': 2.0, 'edge': SUCCESS, 'fill': '#eefbf1', 'title': 'Validation metrics', 'lines': [
            'Accuracy, weighted F1, ROC-AUC',
            'Computed on holdout test set',
            'Cross-validation mean and variation reported',
        ]},
        {'x': 0.5, 'y': 1.95, 'w': 3.8, 'h': 2.0, 'edge': PRIMARY, 'fill': '#fff1f3', 'title': 'ROC / confusion matrix', 'lines': [
            'ROC curve shows ranking quality',
            'Confusion matrix shows error pattern',
            'Useful for threshold interpretation',
        ]},
        {'x': 4.8, 'y': 4.65, 'w': 4.0, 'h': 2.0, 'edge': '#2f7a57', 'fill': '#effbf6', 'title': 'SHAP KernelExplainer', 'lines': [
            'Global feature importance',
            'Top predictors ranked by mean |SHAP|',
            'Used offline for cohort-level understanding',
        ]},
        {'x': 4.8, 'y': 1.95, 'w': 4.0, 'h': 2.0, 'edge': '#1f6b3f', 'fill': '#eefbf1', 'title': 'LIME LimeTabular', 'lines': [
            'Local patient-level explanation',
            'Shows which features drove one result',
            'Returned with each explanation-enabled prediction',
        ]},
        {'x': 9.4, 'y': 3.3, 'w': 3.95, 'h': 2.0, 'edge': DARK, 'fill': '#faf7f8', 'title': 'Interpretation message', 'lines': [
            'Model output is educational, not diagnostic',
            'Explains patterns, not certainty',
            'Supports informed clinical discussion',
        ]},
    ],
    arrows=[
        {'from': (4.3, 5.65), 'to': (4.8, 5.65), 'color': SUCCESS},
        {'from': (4.3, 2.95), 'to': (4.8, 2.95), 'color': PRIMARY},
        {'from': (8.8, 5.65), 'to': (9.4, 4.3), 'color': '#2f7a57'},
        {'from': (8.8, 2.95), 'to': (9.4, 3.75), 'color': '#1f6b3f'},
    ],
    figsize=(14, 8),
    xlim=(0, 14),
    ylim=(0, 8),
)

# 7C-4 Deployment and privacy overview
save_box_diagram(
    'deployment_privacy_overview.png',
    'EndoWherAI — Deployment and Privacy Architecture',
    'Web app stack, backend services, data storage, and security controls',
    [
        {'x': 0.4, 'y': 4.7, 'w': 3.4, 'h': 2.0, 'edge': '#1a3a80', 'fill': '#eef4ff', 'title': 'Frontend', 'lines': [
            'Next.js 14',
            'TypeScript + Tailwind CSS v4',
            'User interface and visual analytics',
        ]},
        {'x': 0.4, 'y': 2.0, 'w': 3.4, 'h': 1.95, 'edge': '#9a7020', 'fill': '#fff8e0', 'title': 'Backend API', 'lines': [
            'FastAPI routes',
            'Prediction, insights, chat, remedies',
            'JWT-secured requests',
        ]},
        {'x': 4.4, 'y': 3.35, 'w': 3.55, 'h': 2.1, 'edge': '#7a5510', 'fill': '#faf2df', 'title': 'Data layer', 'lines': [
            'Supabase / PostgreSQL',
            'User profiles, symptom logs, cycles',
            'Row-level security enabled',
        ]},
        {'x': 8.35, 'y': 4.7, 'w': 5.2, 'h': 2.0, 'edge': '#1a6640', 'fill': '#e8fff0', 'title': 'Privacy controls', 'lines': [
            'Pseudonymisation per EDPB 01/2025',
            'No direct identifiers stored',
            'Scoped access with RLS and auth tokens',
        ]},
        {'x': 8.35, 'y': 2.0, 'w': 5.2, 'h': 1.95, 'edge': DARK, 'fill': '#faf7f8', 'title': 'Deployment outcome', 'lines': [
            'Secure, privacy-aware health platform',
            'Supports tracking, insights, and explanations',
            'Built for educational decision support',
        ]},
    ],
    arrows=[
        {'from': (3.8, 5.65), 'to': (4.4, 4.45), 'color': '#1a3a80'},
        {'from': (3.8, 2.85), 'to': (4.4, 3.95), 'color': '#9a7020'},
        {'from': (7.95, 4.4), 'to': (8.35, 5.05), 'color': '#7a5510'},
        {'from': (7.95, 3.0), 'to': (8.35, 2.95), 'color': '#1a6640'},
    ],
    figsize=(14, 8),
    xlim=(0, 14),
    ylim=(0, 8),
)

Saved → /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs/data_preprocessing_overview.png
Saved → /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs/model_architecture_overview.png
Saved → /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs/validation_explainability_overview.png
Saved → /Users/edinabajric/Desktop/Codes/AI_ML/EndowherAI/machine-learning/outputs/deployment_privacy_overview.png


## 8 — Summary: all generated files

In [30]:
print('=== Outputs generated ===')
for f in sorted(os.listdir(OUT_DIR)):
    path = os.path.join(OUT_DIR, f)
    size = os.path.getsize(path) // 1024
    print(f'  {f:<45}  {size} KB')

=== Outputs generated ===
  architecture_diagram.png                       294 KB
  class_distribution.png                         62 KB
  community_insights.json                        16 KB
  confusion_matrix.png                           39 KB
  endowher_stacking_model.joblib                 1982 KB
  lime_local_explanation.html                    1204 KB
  lime_local_explanation.png                     117 KB
  performance_metrics.png                        68 KB
  poster_summary_infographic.png                 471 KB
  roc_curve.png                                  81 KB
  shap_beeswarm.png                              158 KB
  shap_global_summary.png                        69 KB
  tech_stack.png                                 240 KB
